# Lighter-weight Fact Verification Approach ->  Chunk-level

This notebook evalutes the process of using chunk-level candidates for the fact verification. 
What this notebook does:

1. Creates the chunks and loads the annotated data
2. Perform NER pre-filter
3. Examine the exact-match results
4. Examine String-based comparison results + False positive cases
5. Examine Embedding similarity results + False positive cases
6. Examine Paraphrase detection results + False positive cases
7. Examine NLI results for both small and large models + False positive cases

The helper functions are inside `chunk_fact_check_utils.py` (imported as `fc`).


In [ ]:
import os
import pandas as pd
import numpy as np
import pickle
import spacy
from sklearn.metrics import precision_score, recall_score, f1_score, classification_report, confusion_matrix, ConfusionMatrixDisplay
from sentence_transformers import SentenceTransformer, CrossEncoder
from sklearn.metrics.pairwise import cosine_similarity
from scipy.special import softmax
import csv
from tqdm import tqdm

import chunk_fact_check_utils as fc

## Paths & chunk segmentation

In [ ]:
# Sentence splitter + token counter for chunking (passed into fc.sent_chunk_reading)
chunk_nlp = spacy.load("en_core_web_sm")
chunk_nlp.max_length = 3_000_000
chunk_tok_model = SentenceTransformer("all-MiniLM-L6-v2")   # tokenizer used for token counts
count_tokens = lambda s: len(chunk_tok_model.tokenizer.tokenize(s))

In [ ]:
input_dir = r"...\text_test"
output_dir = r"...\segmented"
os.makedirs(output_dir, exist_ok=True)

for fname in os.listdir(input_dir):
    if not fname.endswith(".txt"):
        continue
    in_path = os.path.join(input_dir, fname)
    out_path = os.path.join(output_dir, fname)

    chunks = fc.sent_chunk_reading(in_path, chunk_nlp, count_tokens,
                                   target_tokens=150, max_tokens=250, overlap_tokens=30)
    with open(out_path, "w", encoding="utf-8") as f:
        for chunk in chunks:
            f.write(chunk + "\n")
    print(f"  Segmented: {fname} ({len(chunks)} chunks)")

## 1. Load annotated data

In [ ]:
file_path = r"...\annotated_facts_2026-03-31.json"

full_data_json = fc.read_json(file_path)
info_results_json = fc.json_feature(full_data_json)

all_facts   = fc.build_all_facts (info_results_json)
gold_labels = fc.build_gold_labels(info_results_json)   # shared by every method: 1=verified, 0=contradicted
print(f"{len(all_facts)} facts present")

## 2. NER preprocessing (on chunks)

`fc.ner_per_file` reads one item per line — here each line is a *chunk* — and tags it, which matches the old chunk-reading loop exactly. Pass the transformer NER model in.

In [ ]:
nlp_trans = spacy.load("en_core_web_trf", disable=["tagger", "parser", "lemmatizer", "attribute_ruler"])
nlp_trans_lg  = spacy.load("en_core_web_lg",  disable=["tagger", "parser", "lemmatizer", "attribute_ruler"])

In [ ]:
# build the NER cache, then save it. Then just load the pickle below.
per_file_ner_chunk = fc.ner_per_file(info_results_json, output_dir, nlp_trans)
with open("new_chunk_ner.pkl", "wb") as ner_sents:
    pickle.dump(per_file_ner_chunk, ner_sents)

In [ ]:
with open("new_chunk_ner.pkl", "rb") as ner_sents:
    per_file_ner_chunk = pickle.load(ner_sents)
per_file_ner_chunk

In [ ]:
# keep only chunks that contain at least one named entity
per_file_ner_chunk_filtered = fc.filter_ner_sentences(per_file_ner_chunk)
per_file_ner_chunk_filtered

In [ ]:
file_path = r"...\segmented"

## 3. Exact-match baseline

In [ ]:
# fc.exact_match_score = substring exact match (1 if fact appears verbatim in a sentence)
exact_scores = []
gold_labels = gold_labels 

for info in info_results_json:
    actual_path = os.path.join(file_path, info["text_name"])
    if actual_path not in per_file_ner_chunk:
        print("WRONG FILE")

    sentences = per_file_ner_chunk_filtered[actual_path]
    score = fc.exact_match_score(info["fact"], sentences)
    exact_scores.append(score)

    if score == 1:
        print(f"MATCH FOUND in {actual_path}")
        print(f"  Fact: {info["fact"]}")
        # Find which sentence matched
        for sent in sentences:
            if info["fact"].strip().lower() in sent.strip().lower():
                print(f"  Sentence: {sent}")
                break
    print()

print(f"\nTotal matches: {sum(exact_scores)} out of {len(exact_scores)}")
print(classification_report(gold_labels, exact_scores, digits=3, target_names=["Contradicted", "Verified"]))

## 4. String-Based —> Levenshtein

In [ ]:
gold_label_lev = []

for i, info in enumerate (info_results_json):
    actual_path = os.path.join(file_path, info["text_name"])
    if actual_path not in per_file_ner_chunk_filtered:
        print("WRONG FILE")
    gold_label_lev.append(1 if info["human_verification_status"] == "verified" else 0)

lev_chosen_thresh = [0.0, 0.45, 0.5, 0.55, 0.6, 0.65, 0.7, 0.75, 0.8, 0.85, 0.9, 0.95, 1]
inspect_thresh_lev = 0.5
fp_inspect_lev = []
for thresh in lev_chosen_thresh:
    predicted_lev = []
    fp_sentences_lev = []
    for i, info in enumerate (info_results_json):
        actual_path = os.path.join(file_path, info["text_name"])
        if actual_path not in per_file_ner_chunk_filtered:
            print("WRONG FILE")
   
        label_lev, score_lev, sentence_lev = fc.get_best_match_ner_lev(
            info["fact"],
            per_file_ner_chunk_filtered[actual_path],
            "seq_ratio",
            thresh

        )
        predicted_lev.append(label_lev)
        if label_lev == 1 and gold_label_lev[i] == 0:
            fp_sentences_lev.append((info["fact"], sentence_lev, score_lev))
    if thresh == inspect_thresh_lev:
        fp_inspect_lev = fp_sentences_lev

    fc.report_at_threshold(gold_label_lev, predicted_lev, thresh)

# To inspect false positives at the chosen threshold:
# for fact, sent, sc in fp_inspect_lev:
#     print(f"score={sc}\n  fact: {fact}\n  sent: {sent}\n" + "-" * 80)

## 5. String-Based —> RapidFuzz

In [ ]:
gold_label_rf = []

for i, info in enumerate (info_results_json):
    actual_path = os.path.join(file_path, info["text_name"])
    if actual_path not in per_file_ner_chunk_filtered:
        print("WRONG FILE")
    gold_label_rf.append(1 if info["human_verification_status"] == "verified" else 0)

rf_chosen_thresh = [0, 40, 45, 50, 55, 60, 65, 70, 75, 80, 85, 90, 100]
inspect_thresh_rf = 55
fp_inspect_rf = []
for thresh in rf_chosen_thresh:
    predicted_rf = []
    fp_sentences_rf = []
    for i, info in enumerate (info_results_json):
        actual_path = os.path.join(file_path, info["text_name"])
        if actual_path not in per_file_ner_chunk_filtered:
            print("WRONG FILE")

        label_rf, score_rf, sentence_rf = fc.get_best_match_ner_rapfuz(
            info["fact"],
            per_file_ner_chunk_filtered[actual_path],
            "token_set",
            thresh

        )
        predicted_rf.append(label_rf)
        if label_rf == 1 and gold_label_rf[i] == 0:
            fp_sentences_rf.append((info["fact"], sentence_rf, score_rf))
    if thresh == inspect_thresh_rf:
        fp_inspect_rf = fp_sentences_rf

    fc.report_at_threshold(gold_label_rf, predicted_rf, thresh)

# To inspect false positives at the chosen threshold:
# for fact, sent, sc in fp_inspect_rf:
#     print(f"score={sc}\n  fact: {fact}\n  sent: {sent}\n" + "-" * 80)

## 6. Embedding Similarity approach —> all-MiniLM-L6-v2

In [ ]:
model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

In [ ]:
embedding_cache_emb_chunk = fc.build_embedding_cache(per_file_ner_chunk_filtered, model)
with open("new_embedding_cache_emb.pkl", "wb") as f:
    pickle.dump(embedding_cache_emb_chunk, f)

In [ ]:
chosen_thresh_emb = [0.0, 0.2, 0.5, 0.55, 0.57, 0.59, 0.6, 0.62, 0.63, 0.64, 0.65, 0.67, 0.7, 0.75, 0.8, 0.85, 0.9, 1]
inspect_thresh_emb = 0.65
fp_inspect_emb = []
for thresh in chosen_thresh_emb:
    emb_predicted = []
    gold_label_emb = []
    fp_sentences_emb = []
    for i, info in enumerate(info_results_json):
        actual_path = os.path.join(file_path, info["text_name"])
        if actual_path not in per_file_ner_chunk_filtered:
            print("WRONG FILE")

        score_emb, matched_sentence_emb, found_emb = fc.get_first_above_thresh(
            info["fact"],
            per_file_ner_chunk_filtered[actual_path],
            embedding_cache_emb_chunk[actual_path],
            model,
            thresh
        )
        gold_label_emb.append(1 if info["human_verification_status"] == "verified" else 0)
        emb_predicted.append(1 if found_emb else 0)
        if found_emb and gold_label_emb[i] == 0:
            fp_sentences_emb.append((info["fact"], matched_sentence_emb, score_emb))

    if thresh == inspect_thresh_emb:
        fp_inspect_emb = fp_sentences_emb

    fc.report_at_threshold(gold_label_emb, emb_predicted, thresh)
# To inspect false positives at the chosen threshold:
# for fact, sent, sc in fp_inspect_emb:
#     print(f"score={sc}\n  fact: {fact}\n  sent: {sent}\n" + "-" * 80)

## 8. Paraphrase Detection —> paraphrase-mpnet-base-v2

In [ ]:
model_paraph = SentenceTransformer("sentence-transformers/paraphrase-mpnet-base-v2")

In [ ]:
# APPROACH B cache (own variable: embedding_cache_paraph)
embedding_cache_paraph = fc.build_embedding_cache(per_file_ner_chunk_filtered, model_paraph, show_progress_bar=True)
with open("new_embedding_cache_chunk_paraph.pkl", "wb") as f:
    pickle.dump(embedding_cache_paraph, f)

In [ ]:
chosen_thresh_paraph = [0.0, 0.2, 0.5, 0.55, 0.57, 0.59, 0.6, 0.62, 0.63, 0.64, 0.65, 0.67, 0.7, 0.75, 0.8, 0.9, 1]
inspect_thresh_paraph = 0.64
fp_inspect_paraph = []
for thresh in chosen_thresh_paraph:
    paraph_predicted = []
    gold_label_paraph = []
    fp_sentences_paraph = []
    for i, info in enumerate(info_results_json):
        actual_path = os.path.join(file_path, info["text_name"])
        if actual_path not in per_file_ner_chunk_filtered:
            print("WRONG FILE")

        score_paraph, matched_sentence_paraph, found_paraph = fc.get_first_above_thresh_paraph(
            info["fact"],
            per_file_ner_chunk_filtered[actual_path],
            embedding_cache_paraph[actual_path],
            model_paraph,
            thresh
        )
        gold_label_paraph.append(1 if info["human_verification_status"] == "verified" else 0)
        paraph_predicted.append(1 if found_paraph else 0)
        if found_paraph and gold_label_paraph[i] == 0:
            fp_sentences_paraph.append((info["fact"], matched_sentence_paraph, score_paraph))

    if thresh == inspect_thresh_paraph:
        fp_inspect_paraph = fp_sentences_paraph

    fc.report_at_threshold(gold_label_paraph, paraph_predicted, thresh)
# To inspect false positives at the chosen threshold:
# for fact, sent, sc in fp_inspect_paraph:
#     print(f"score={sc}\n  fact: {fact}\n  sent: {sent}\n" + "-" * 80)

## 9. NLI cross-encoder

The stored scores are **probabilities**. Because of that, every decision/approach
function below indexes the probabilities directly (`s[E]`, `s[C]`, ...) and is kept local. Label order for `nli-deberta` is
`E, N, C = 1, 2, 0` that matches `fc.NLI_DEBERTA`, and `E, N, C = 0, 1, 2` that matches `fc.MORITZLAURER`. 

The cell below represent how the scores and matches candidates were captured as an example. To prevent confusion, the value between 



In [ ]:
model_cross = CrossEncoder('cross-encoder/nli-deberta-v3-base')
model_cross_advanc = CrossEncoder("MoritzLaurer/DeBERTa-v3-base-mnli-fever-anli")

In [ ]:
# building the probability checkpoint. 
# using tqdm to show the progress and pickle to save each 20 facts scores, to save the
# progress in case of computer crash.
# This cell is similar for both small and large NLI models.

checkpoint_file_small = "nli_small_chunk_with_softmax.pkl"
try:
    with open(checkpoint_file_small, "rb") as f:
        checkpoint = pickle.load(f)
    all_fact_data = checkpoint["all_fact_data"]
    gold_labels = checkpoint["gold_labels"]
    start_idx = checkpoint["next_idx"]
    print(f"Resuming from fact {start_idx}")
except FileNotFoundError:
    all_fact_data, gold_labels, start_idx = [], [], 0
    print("Starting fresh")

for i in tqdm(range(start_idx, len(info_results_json)), desc="Facts"):
    info = info_results_json[i]
    actual_path = os.path.join(file_path, info["text_name"])
    if actual_path not in per_file_ner_chunk_filtered:
        print(f"PATH NOT FOUND for fact {i}: {actual_path}")
        continue
    all_scores, all_sentences = fc.get_all_pairs_cross_encoder(
        all_facts[i], per_file_ner_chunk_filtered[actual_path],
        model_cross, apply_softmax=True          # stores probabilities, therefore no need to add softmax later
    )
    all_fact_data.append({"fact_idx": i, "fact": info["fact"],
                          "document": info["text_name"],
                          "sentences": all_sentences, "scores": all_scores})
    gold_labels.append(1 if info["human_verification_status"] == "verified" else 0)
    if len(all_fact_data) % 20 == 0:
        with open(checkpoint_file_small, "wb") as f:
            pickle.dump({"all_fact_data": all_fact_data, "gold_labels": gold_labels,
                         "next_idx": i + 1}, f)

with open(checkpoint_file_small, "wb") as f:
    pickle.dump({"all_fact_data": all_fact_data, "gold_labels": gold_labels,
                 "next_idx": len(info_results_json)}, f)
print(f"\nDone. {len(all_fact_data)} facts processed.")

### 9a. Small Model —> Early-Exit & Asymmetric

In [ ]:
with open("nli_small_chunk_with_softmax.pkl", "rb") as f:    
    checkpoint = pickle.load(f)
all_fact_data = checkpoint["all_fact_data"]
gold_labels_all_cross = checkpoint["gold_labels"]


print("First pair scores:", all_fact_data[0]["scores"][0])
print("Sum:", sum(all_fact_data[0]["scores"][0]))  # check the output to make sure they are probabilities

thresholds = [0.0, 0.1, 0.2, 0.25, 0.3, 0.35, 0.4, 0.45, 0.5, 0.55, 0.6, 0.65, 0.7, 0.77, 0.8, 0.85, 0.9, 0.93, 0.95, 0.96, 0.97, 0.98, 0.99, 1, 1.1]
# nli-deberta :{0: 'contradiction', 1: 'entailment', 2: 'neutral'}

E, N, C = fc.NLI_DEBERTA["entailment"], fc.NLI_DEBERTA["neutral"], fc.NLI_DEBERTA["contradiction"]
for thresh in thresholds:
    preds = [fc.predict_early_exit(entry, thresh, E)
                for entry in all_fact_data]
    #preds = [fc.predict_asymmetric(entry, thresh, E, C) for entry in all_fact_data]
    fc.report_at_threshold(gold_labels_all_cross, preds, thresh, digits=3)

### 9b. Large Model —> Early-Exit & Asymmetric

In [ ]:
with open("nli_large_chunk_with_softmax.pkl", "rb") as f:    
    checkpoint_2 = pickle.load(f)
all_fact_data_2 = checkpoint_2["all_fact_data"]
gold_labels_all_cross_2 = checkpoint_2["gold_labels"]

#  stored values probabilities 
print("First pair scores:", all_fact_data_2[0]["scores"][0])
print("Sum:", sum(all_fact_data_2[0]["scores"][0]))  # check the output to make sure they are probabilities

thresholds = [0.0, 0.1, 0.2, 0.25, 0.3, 0.35, 0.4, 0.45, 0.5, 0.55, 0.6, 0.65, 0.7, 0.77, 0.8, 0.85, 0.9, 0.93, 0.95, 0.96, 0.97, 0.98, 0.99, 1, 1.1]
# MoritzLaurer model: {0: entailment, 1: neutral, 2: contradiction}

E, N, C = fc.MORITZLAURER["entailment"], fc.MORITZLAURER["neutral"], fc.MORITZLAURER["contradiction"]
for thresh in thresholds:
    preds_l = [fc.predict_early_exit(entry, thresh, E)
                    for entry in all_fact_data_2]
    #preds_l = [fc.predict_asymmetric(entry, thresh, E, C) for entry in all_fact_data_2]
    fc.report_at_threshold(gold_labels_all_cross_2, preds_l, thresh, digits=3)

### 9c. False Positive Inspection

In [ ]:
threshold = 0.9
E, N, C = fc.NLI_DEBERTA["entailment"], fc.NLI_DEBERTA["neutral"], fc.NLI_DEBERTA["contradiction"]

fp = 0
for entry, gold in zip(all_fact_data, gold_labels_all_cross):
    pred, chunk, score = fc.early_exit_decision(entry, threshold, E)

    if pred == 1 and gold == 0:
        fp += 1
  
        print(f"[FP (said Verified, was Contradicted)]  E={score[E]:.3f} N={score[N]:.3f} C={score[C]:.3f}")
        print(f"  Fact:  {entry['fact']}")
        print(f"  Chunk: {chunk[:200]}")
        print("-" * 80)

print(f"\nTotal false positives: {fp}")

In [ ]:
E, N, C = fc.MORITZLAURER["entailment"], fc.MORITZLAURER["neutral"], fc.MORITZLAURER["contradiction"]

thresh = 0.95
save_path = r".../nli_false_positive_adv_chunk.csv"

with open(save_path, "w", newline="", encoding="utf-8") as fcsv:
    writer = csv.writer(fcsv)
    writer.writerow(["fact", "exit_chunk", "entail_prob"])

    for i, e in enumerate(all_fact_data_2):
        for j, s in enumerate(e["scores"]):
            p = s[E]
            if p >= thresh:
                if gold_labels_all_cross_2[i] == 0:
                    writer.writerow([e["fact"], e["sentences"][j], round(float(p), 4)])
                break

print("CSV SAVED")